# JupyterLite Contents API

This notebook demonstrates the browser-native `jupyterlite.contents` helper exposed by the JavaScript kernel.

It works in both `JavaScript` and `JavaScript (Worker)`, so you can switch kernels and rerun the same cells.

## Set Up A Demo Directory

Relative paths resolve from the notebook directory. This notebook creates a temporary `contents-demo/` folder next to itself, uses it for the examples, then removes it at the end.

In [ ]:
const contents = jupyterlite.contents;
const demoDir = 'contents-demo';

await contents.mkdir(demoDir, { recursive: true });

const setupSummary = {
    cwd: contents.cwd(),
    demoDir: contents.resolve(demoDir),
    notebookEntries: await contents.listdir('.')
};

setupSummary

## Write Text, JSON, And Bytes

The helper chooses the storage format explicitly through `writeText()`, `writeJSON()`, and `writeBytes()`.

In [ ]:
await contents.writeText(
    `${demoDir}/notes.txt`,
    [
        'Created from a JavaScript notebook.',
        `cwd: ${contents.cwd()}`,
        `created: ${new Date().toISOString()}`
    ].join('\n')
);

await contents.writeJSON(`${demoDir}/data.json`, {
    answer: 42,
    values: [1, 1, 2, 3, 5, 8]
});

await contents.writeBytes(`${demoDir}/tag.bin`, new Uint8Array([74, 83, 75]));

const writeSummary = {
    entries: await contents.listdir(demoDir),
    data: await contents.readJSON(`${demoDir}/data.json`),
    tag: Array.from(await contents.readBytes(`${demoDir}/tag.bin`))
};

writeSummary

## Inspect File Metadata

Use `stat()` to fetch lightweight information for files and directories.

In [ ]:
const stats = await Promise.all(
    (await contents.listdir(demoDir)).map(name => contents.stat(`${demoDir}/${name}`))
);

stats

## Rename And Verify

Paths stay notebook-relative unless you pass an absolute drive path.

In [ ]:
await contents.rename(`${demoDir}/data.json`, `${demoDir}/payload.json`);

const renameSummary = {
    exists: await contents.exists(`${demoDir}/payload.json`),
    textPreview: await contents.readText(`${demoDir}/notes.txt`),
    entriesAfterRename: await contents.listdir(demoDir)
};

renameSummary

## Clean Up

This removes the temporary files created by the demo.

In [ ]:
for (const name of await contents.listdir(demoDir)) {
    await contents.remove(`${demoDir}/${name}`);
}

await contents.remove(demoDir);

const cleanupSummary = {
    demoDirRemoved: !(await contents.exists(demoDir)),
    notebookEntries: await contents.listdir('.')
};

cleanupSummary